In [1]:
# -*- coding: utf-8 -*-
# --- Cell 1: Imports and Setup (edX Only) ---

import time
import re
from datetime import datetime
import pandas as pd
import unicodedata

# Selenium Imports
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By # Import By for locating elements
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException, WebDriverException

# Webdriver Manager Imports
from webdriver_manager.chrome import ChromeDriverManager

print("[Setup] Importing libraries...")

# --- Configuration ---

# !!! Using ONLY the edX URLs !!!
QUORA_LOG_URLS = [
    'https://www.quora.com/Are-certificates-from-Udemy-edX-and-Coursera-of-any-worth/log',
    'https://www.quora.com/What-is-the-value-of-edX-certificates/log',
    'https://www.quora.com/What-are-the-best-data-science-courses-on-edX/log',
    'https://www.quora.com/Which-is-better-Coursera-Udacity-or-edX/log',
    'https://www.quora.com/Is-it-worth-it-getting-the-Harvards-Data-Science-Professional-Certificate-on-EDX/log',
    'https://www.quora.com/Which-is-the-best-course-for-learning-algorithms-and-data-structure-on-Coursera-edX-or-Udemy/log',
    'https://www.quora.com/Should-I-mention-Coursera-edX-and-Udacity-certificates-in-my-resume/log',
    'https://www.quora.com/Is-it-worth-studying-Data-Science-MicroMaster-on-Edx/log',
    'https://www.quora.com/Which-is-the-best-online-site-to-learn-data-science-EdX-DataCamp-or-Coursera/log',
    'https://www.quora.com/Which-has-the-best-CS101-Introduction-to-Computer-Science-course-Coursera-Udacity-or-edX/log',
    'https://www.quora.com/How-do-courses-on-Udemy-compare-to-those-on-EdX-and-Coursera/log',
    'https://www.quora.com/Is-doing-free-edX-courses-worth-it/log',
    'https://doronaxeton.quora.com/Which-is-better-Coursera-Udacity-or-edX/log',
    'https://www.quora.com/Are-there-any-free-certification-options-for-Data-Science-and-Machine-Learning-on-Coursera-or-Edx/log',
    'https://www.quora.com/What-led-to-Google-ending-its-partnership-with-edX/log',
    'https://www.quora.com/Are-there-any-free-courses-offered-by-top-universities-such-as-Harvard-and-MIT-on-edX/log',
    'https://www.quora.com/Should-certificates-earned-through-edX-be-included-on-a-CV-or-LinkedIn-profile/log',
    'https://www.quora.com/What-are-some-examples-of-online-universities-that-offer-accredited-degrees-such-as-Coursera-and-edX/log',
    'https://www.quora.com/Why-is-18-03x-differential-equations-not-available-in-edX/log',
    'https://www.quora.com/Do-employers-value-free-courses-from-Coursera-and-EdX/log',
    'https://www.quora.com/Can-Coursera-and-edX-courses-help-you-get-accepted-to-top-universities-like-Harvard-MIT-Oxford-etc/log',
    'https://www.quora.com/How-do-EdX-courses-compare-to-courses-offered-directly-by-universities-on-their-own-websites/log',
    'https://www.quora.com/Which-website-is-actually-really-useful-for-doing-an-online-course-Coursera-or-edX/log',
    'https://www.quora.com/Is-EdX-a-scam-or-a-legit-learning-platform/log',
    'https://www.quora.com/Is-edx-free/log',
    'https://www.quora.com/Does-certificates-from-websites-like-Coursera-Udacity-or-Edx-help-you-get-a-job/log',
    'https://www.quora.com/What-is-the-best-course-specialization-to-take-on-Coursera-Udacity-or-EdX-for-Data-Science/log',
    'https://www.quora.com/Is-it-worth-getting-a-verified-certificate-from-CS50-on-edX/log',
    'https://www.quora.com/How-are-MITx-statistics-and-data-science-micromasters-programs-on-edX/log',
    'https://www.quora.com/Are-edX-certificates-valuable-to-universities/log',
    'https://www.quora.com/Has-anyone-improved-their-job-prospects-after-taking-courses-from-Coursera-edX-or-other-online-training/log',
    'https://www.quora.com/What-are-the-best-courses-for-a-beginner-in-data-science-or-machine-learning-on-Coursera-and-edX-Which-platform-Coursera-or-edX-would-be-more-suitable-for-a-beginner-in-these-fields/log',
    'https://fionnedgard.quora.com/Does-having-verified-course-certificates-from-edX-Coursera-etc-help-in-Ph-D-admissions/log',
    'https://www.quora.com/Can-I-pursue-a-masters-degree-in-software-engineering-after-a-bachelors-degree-in-chemical-engineering-I-just-got-2-years-experience-in-this-field-and-I-got-a-lot-of-certificates-Microsoft-Coursera-edX-etc-for-this/log',
    'https://www.quora.com/What-are-the-reasons-for-people-choosing-Coursera-EdX-and-Udacity-over-Udemy-for-online-courses/log',
    'https://www.quora.com/Can-edX-certificates-be-used-for-job-applications-at-all-companies-including-IBM-worldwide/log',
    'https://www.quora.com/What-are-the-differences-between-an-EdX-certificate-and-a-Coursera-certificate-and-which-one-is-more-beneficial-for-adding-to-a-LinkedIn-profile/log',
    'https://www.quora.com/Do-Coursera-and-edX-certificates-hold-value-on-resumes-CVs-even-if-they-are-free-and-not-paid-for/log',
    'https://www.quora.com/Can-completing-courses-from-Coursera-and-Edx-lead-to-job-opportunities-with-companies-like-Amazon-and-Microsoft/log',
    'https://www.quora.com/How-can-one-verify-the-authenticity-of-an-education-degree-obtained-through-distance-learning-such-as-EdX-courses-MOOC/log'
]


MAX_ANSWERS_PER_URL = 100 # Keep limit per URL
# !!! Setting specific output filename for edX !!!
OUTPUT_FILENAME = 'edx_reviews_log.csv' # Changed filename for edX
BASE_URL = 'https://www.quora.com/' # For login

# --- Initialize Data Storage ---
all_answers_data = [] # Initialize list to store data

print("[Setup] Imports and configuration complete.")
print(f"[Setup] Target /log URLs count: {len(QUORA_LOG_URLS)}") # Show count (should be 40)
print(f"[Setup] Output file: {OUTPUT_FILENAME}")
print("[Setup] NOTE: This version scrapes /log pages and saves Full Text & Date.")

[Setup] Importing libraries...
[Setup] Imports and configuration complete.
[Setup] Target /log URLs count: 40
[Setup] Output file: edx_reviews_log.csv
[Setup] NOTE: This version scrapes /log pages and saves Full Text & Date.


In [2]:
# -*- coding: utf-8 -*-
# --- Cell 2: Function to Initialize Selenium WebDriver ---
# (Code remains the same as in your provided notebook)

def initialize_driver():
    """ Initializes Selenium Chrome WebDriver. """
    try:
        print("[WebDriver] Initializing Chrome WebDriver...")
        options = webdriver.ChromeOptions()
        options.add_argument('--disable-gpu')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument("--disable-notifications") # Try disabling browser notifications
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36") # Example: Update to a recent Chrome version
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
        driver.maximize_window()
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        print("[WebDriver] WebDriver initialized successfully.")
        return driver
    except WebDriverException as e:
         print(f"[WebDriver Error] WebDriver setup issue: {e}. Ensure Chrome/Chromedriver is installed and compatible.")
         return None
    except Exception as e:
        print(f"[WebDriver Error] Error initializing WebDriver: {e}")
        return None

In [3]:
# -*- coding: utf-8 -*-
# --- Cell 3: Manual Login Step ---
# (Code remains the same as in your provided notebook)

driver = None

try:
    driver = initialize_driver()

    if driver:
        print(f"\n[Login] Navigating to Quora homepage: {BASE_URL}")
        driver.get(BASE_URL)
        print("\n>>> ACTION REQUIRED <<<")
        print(">>> A Chrome browser window controlled by Selenium has opened.")
        print(">>> Please MANUALLY log in to your Quora account in THAT window.")
        print(">>> Wait for the main Quora feed/homepage to load after successful login.")

        input("\n>>> Press Enter in this cell AFTER you have successfully logged in manually to continue...")

        print("\n[Login] Resuming script. Assuming manual login was successful.")
        time.sleep(3)

        try:
             post_login_selector = "#root > div > div.q-box > div > div.q-fixed.qu-fullX.qu-zIndex--header.qu-bg--raised.qu-borderBottom.qu-boxShadow--medium.qu-borderColor--raised > div > div:nth-child(2) > div > div:nth-child(8) > div > div > div > div > div > div.q-relative.qu-display--inline-flex.qu-alignItems--center.qu-justifyContent--center > div > div > div > div.q-box.qu-borderRadius--circle.qu-borderAll.qu-borderColor--darken" # <-- USER PROVIDED (FRAGILE)
             print(f"[Login Verification] Attempting to verify login using selector: '{post_login_selector[:50]}...'")

             WebDriverWait(driver, 15).until(
                 EC.presence_of_element_located((By.CSS_SELECTOR, post_login_selector))
             )
             print("[Login Verification] Logged-in state element found. Proceeding...")
        except TimeoutException:
             print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
             print("[Login Verification WARNING] Could NOT verify logged-in state using the provided selector.")
             print("The script WILL CONTINUE, but scraping WILL LIKELY FAIL if not logged in.")
             print("Consider finding a more reliable selector for the login check (e.g., based on profile link/icon class).")
             print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")
        except Exception as verify_err:
             print(f"[Login Verification ERROR] Unexpected error during login check: {verify_err}")

    else:
        print("[Login] WebDriver initialization failed. Cannot proceed.")
        driver = None
        raise Exception("WebDriver failed to initialize.")

except Exception as e:
    print(f"[Login Error] An error occurred during the manual login step setup: {e}")
    if driver:
        driver.quit()
    driver = None
    raise Exception(f"Error during login setup: {e}")

# If successful, the 'driver' variable holds the logged-in browser session.

[WebDriver] Initializing Chrome WebDriver...
[WebDriver] WebDriver initialized successfully.

[Login] Navigating to Quora homepage: https://www.quora.com/

>>> ACTION REQUIRED <<<
>>> A Chrome browser window controlled by Selenium has opened.
>>> Please MANUALLY log in to your Quora account in THAT window.
>>> Wait for the main Quora feed/homepage to load after successful login.



>>> Press Enter in this cell AFTER you have successfully logged in manually to continue... 



[Login] Resuming script. Assuming manual login was successful.
[Login Verification] Attempting to verify login using selector: '#root > div > div.q-box > div > div.q-fixed.qu-ful...'
[Login Verification] Logged-in state element found. Proceeding...


In [4]:
# -*- coding: utf-8 -*-
# --- Cell 4: Extract Full Text Function (Log Page) - REPAIRED ---
# (Code remains the same as in your provided notebook)

def extract_full_text_log(answer_container_element, driver, wait):
    """
    Extracts the full answer text AND the revision date/time string
    from within a given answer container WebElement on a /log page,
    repeatedly clicking 'more' buttons until none are found.

    Args:
        answer_container_element (WebElement): The Selenium WebElement for the answer container.
        driver (webdriver): The Selenium WebDriver instance.
        wait (WebDriverWait): The WebDriverWait instance.

    Returns:
        tuple: (full_answer_text, date_str)
    """
    full_answer_text = None
    date_str = 'N/A'
    max_clicks = 5

    bordered_text_box_selector = "div.q-box.qu-borderRadius--medium.qu-p--medium.qu-mb--medium.qu-borderAll"
    more_button_selector = "div.qt_read_more"
    text_paragraph_selector = "p.q-text"
    metadata_footer_selector = "div.q-text.qu-dynamicFontSize--small.qu-mt--small.qu-color--gray_light.qu-passColorToLinks"
    footer_spans_selector = "span.c1h7helg"

    try:
        try:
            metadata_footer = answer_container_element.find_element(By.CSS_SELECTOR, metadata_footer_selector)
            footer_spans = metadata_footer.find_elements(By.CSS_SELECTOR, footer_spans_selector)
            if footer_spans:
                potential_date = footer_spans[-1].text.strip()
                if re.search(r'\d{4}', potential_date) and any(month in potential_date for month in ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]):
                     date_str = potential_date
        except NoSuchElementException:
            print(f"      [Date Warn] Metadata footer not found using selector: '{metadata_footer_selector}'. Date will be N/A.")
        except Exception as date_err:
            print(f"      [Date Error] Error extracting date: {date_err}")

        try:
            bordered_text_box = answer_container_element.find_element(By.CSS_SELECTOR, bordered_text_box_selector)
        except NoSuchElementException:
            try:
                 deleted_text_container = answer_container_element.find_element(By.CSS_SELECTOR, "div.q-text.qu-color--gray")
                 deleted_text = deleted_text_container.text.strip()
                 if "deleted by" in deleted_text.lower():
                      # print(f"      [Info] Skipping text extraction for simple deleted message container: '{deleted_text[:50]}...'") # Debug Optional
                      return None, date_str
                 else:
                      # print(f"      [Text Extract Warn] Bordered text box not found but not a recognized deleted message.") # Debug Optional
                      return None, date_str
            except NoSuchElementException:
                  # print(f"      [Text Extract Warn] Neither bordered text box nor simple deleted message found.") # Debug Optional
                  return None, date_str

        clicks = 0
        while clicks < max_clicks:
            try:
                more_button = WebDriverWait(bordered_text_box, 2).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, more_button_selector))
                )
                if more_button and more_button.is_displayed():
                    try:
                        driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});", more_button)
                        time.sleep(0.5)
                        more_button.click()
                        time.sleep(1)
                        clicks += 1
                        bordered_text_box = answer_container_element.find_element(By.CSS_SELECTOR, bordered_text_box_selector)
                    except ElementClickInterceptedException:
                        try:
                            driver.execute_script("arguments[0].click();", more_button)
                            time.sleep(1)
                            clicks += 1
                            bordered_text_box = answer_container_element.find_element(By.CSS_SELECTOR, bordered_text_box_selector)
                        except Exception as js_click_err:
                            break
                    except StaleElementReferenceException:
                         break
                    except Exception as click_err:
                        break
                else:
                    break
            except TimeoutException:
                break
            except NoSuchElementException:
                 break
            except StaleElementReferenceException:
                 break
            except Exception as loop_err:
                break

        try:
            text_paragraphs = bordered_text_box.find_elements(By.CSS_SELECTOR, text_paragraph_selector)
            text_parts = [p.text for p in text_paragraphs if p.text and p.text.strip()]

            if text_parts:
                 full_answer_text = "\n".join(text_parts).strip()
            else:
                 full_answer_text = bordered_text_box.text.strip()

            if full_answer_text:
                 full_answer_text = full_answer_text.replace('(more)', '').strip()
                 if not full_answer_text:
                      full_answer_text = None
            else:
                 full_answer_text = None

        except NoSuchElementException:
            full_answer_text = None
        except StaleElementReferenceException:
             full_answer_text = None
        except Exception as text_err:
            print(f"      [Text Extract Error] Extracting text after clicks: {text_err}")
            full_answer_text = None

    except StaleElementReferenceException:
        return None, date_str
    except Exception as e:
        print(f"      [Error] General error in extract_full_text_log: {e}")
        return None, date_str

    return full_answer_text, date_str

In [5]:
# -*- coding: utf-8 -*-
# --- Cell 5: Main Scraping Logic (Log Page Hybrid) ---
# (Code remains the same as in your provided notebook, using :has() based on bordered box)

if 'driver' in locals() and driver:
    wait = WebDriverWait(driver, 15)

    try:
        print("\n--- Starting Data Extraction Loop (Log Page Hybrid: Full Text Only) ---")
        for url in QUORA_LOG_URLS:
            print(f"\n[INFO] Navigating to: {url}")
            processed_count_for_url = 0
            try:
                driver.get(url)
                log_page_content_selector = "#mainContent"
                try:
                    wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, log_page_content_selector)))
                    print(f"  [INFO] Log page content area ('{log_page_content_selector}') loaded.")
                except TimeoutException:
                    print(f"  [WARN] Timed out waiting for main log page content ('{log_page_content_selector}'). Skipping URL.")
                    continue

                print("  [INFO] Waiting and scrolling /log page (will stop when height stops changing)...")
                scroll_pause_time = 3
                last_height = driver.execute_script("return document.body.scrollHeight")
                no_change_count = 0
                scroll_attempt = 0

                while True:
                    scroll_attempt += 1
                    try:
                        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                        time.sleep(scroll_pause_time)
                        new_height = driver.execute_script("return document.body.scrollHeight")
                    except Exception as scroll_exec_err:
                         print(f"      [Scroll Error] Could not execute scroll script: {scroll_exec_err}")
                         time.sleep(scroll_pause_time)
                         continue

                    if abs(new_height - last_height) < 10:
                        no_change_count += 1
                    else:
                        no_change_count = 0
                    last_height = new_height

                    if no_change_count >= 3:
                         print(f"    Scrolling stopped after {scroll_attempt} attempts.")
                         break
                    if scroll_attempt > 100:
                        print(f"    [WARN] Scrolling stopped after {scroll_attempt} attempts (safety break).")
                        break

                print("  [INFO] Finished scrolling.")
                time.sleep(3)

                answer_container_selector = "#mainContent > div > div > div > div.q-box:has(> div.q-box.qu-borderRadius--medium.qu-p--medium.qu-mb--medium.qu-borderAll)"
                print(f"  [INFO] Finding answer containers using selector: '{answer_container_selector}'")
                answer_container_elements = []
                try:
                    time.sleep(1)
                    answer_container_elements = driver.find_elements(By.CSS_SELECTOR, answer_container_selector)
                    print(f"  [INFO] Found {len(answer_container_elements)} potential answer container elements (using :has()).")
                except NoSuchElementException:
                     print(f"  [WARN] No answer containers found with selector: '{answer_container_selector}'")
                except Exception as find_err:
                    print(f"  [ERROR] Could not find answer containers: {find_err}")
                    print("  [Hint] Ensure your Chrome browser & ChromeDriver support CSS :has() or check selector syntax.")
                    answer_container_elements = []

                if not answer_container_elements:
                     print(f"  [WARN] No answer elements found matching the :has() criteria for {url}. Skipping.")
                     continue

                for i, answer_element in enumerate(answer_container_elements):
                    if processed_count_for_url >= MAX_ANSWERS_PER_URL:
                        print(f"    [INFO] Reached max answers limit ({MAX_ANSWERS_PER_URL}) for this URL.")
                        break

                    # print(f"  [INFO] Processing Answer Element #{i+1} (Matched Container)") # Optional Debug
                    try:
                         driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});", answer_element)
                         time.sleep(0.3)
                    except Exception as scroll_err:
                         pass # Ignore scroll error

                    full_text, extracted_date = extract_full_text_log(answer_element, driver, wait)

                    if full_text:
                        deleted_keywords = ["quora deleted this answer", "author deleted this answer", "deleted by quora moderation"]
                        if any(keyword in full_text.lower() for keyword in deleted_keywords):
                             # print(f"    [INFO] Skipping element #{i+1}: Detected as likely deleted answer text.") # Optional Debug
                             pass
                        else:
                            # print(f"    [OK] Extracted: Date='{extracted_date}' Text='{str(full_text)[:80].replace(chr(10),' ')}...'") # Optional Debug
                            all_answers_data.append({
                                'Question_Log_URL': url,
                                'Full_Answer_Text': full_text,
                                'Date': extracted_date
                            })
                            processed_count_for_url += 1

                if processed_count_for_url == 0:
                    print(f"  [WARN] Found {len(answer_container_elements)} containers matching :has(), but extracted no valid text from {url}. Check selectors inside Cell 4.")

            except WebDriverException as wde:
                 print(f"  [ERROR] WebDriver error processing page {url}: {wde}. Attempting to continue...")
            except Exception as page_error:
                print(f"  [ERROR] General error processing page {url}: {page_error}")

            page_wait_time = 5
            print(f"  [INFO] Waiting {page_wait_time} seconds before next URL...")
            time.sleep(page_wait_time)

    except Exception as main_error:
        print(f"[ERROR] An unexpected error occurred in the main scraping loop: {main_error}")

    finally:
        if 'driver' in locals() and driver:
            print("\n[WebDriver] Closing WebDriver...")
            try:
                driver.quit()
                print("[WebDriver] WebDriver closed.")
            except Exception as quit_err:
                print(f"[WebDriver Error] Error while quitting driver: {quit_err}")
            finally:
                 driver = None
        else:
             print("[WebDriver] WebDriver was not active or already closed.")

else:
    print("\n[Error] Cannot proceed: WebDriver not initialized or manual login step failed/skipped.")

print(f"\n--- Scraping Finished ---")
print(f"Total answers collected (before filtering): {len(all_answers_data)}")
if all_answers_data:
    print("\nPreview of collected data (before filtering):")
    try:
        cols = ['Question_Log_URL', 'Full_Answer_Text', 'Date']
        df_preview = pd.DataFrame(all_answers_data)
        df_preview = df_preview[[c for c in cols if c in df_preview.columns]]
        print(df_preview.head())
    except Exception as df_err:
        print(f"[Error] Could not create DataFrame preview: {df_err}")
else:
    print("\nNo data was collected.")


--- Starting Data Extraction Loop (Log Page Hybrid: Full Text Only) ---

[INFO] Navigating to: https://www.quora.com/Are-certificates-from-Udemy-edX-and-Coursera-of-any-worth/log
  [INFO] Log page content area ('#mainContent') loaded.
  [INFO] Waiting and scrolling /log page (will stop when height stops changing)...
    Scrolling stopped after 60 attempts.
  [INFO] Finished scrolling.
  [INFO] Finding answer containers using selector: '#mainContent > div > div > div > div.q-box:has(> div.q-box.qu-borderRadius--medium.qu-p--medium.qu-mb--medium.qu-borderAll)'
  [INFO] Found 269 potential answer container elements (using :has()).
    [INFO] Reached max answers limit (100) for this URL.
  [INFO] Waiting 5 seconds before next URL...

[INFO] Navigating to: https://www.quora.com/What-is-the-value-of-edX-certificates/log
  [INFO] Log page content area ('#mainContent') loaded.
  [INFO] Waiting and scrolling /log page (will stop when height stops changing)...
    Scrolling stopped after 8 atte

In [6]:
# -*- coding: utf-8 -*-
# --- Cell 6: Filter Data by Year ---
# (Code remains the same as in your provided notebook)

print("\n--- Filtering Data by Year (2024-2025) ---")

filtered_answers_data = []
parse_errors = 0
kept_count = 0
skipped_count = 0

if 'all_answers_data' in locals() and isinstance(all_answers_data, list):
    print(f"Processing {len(all_answers_data)} records...")
    for item in all_answers_data:
        date_str = item.get('Date', 'N/A')
        year = None

        if date_str != 'N/A':
            try:
                datetime_obj = datetime.strptime(date_str, "%B %d, %Y at %I:%M:%S %p")
                year = datetime_obj.year
            except ValueError:
                parse_errors += 1
            except Exception as e:
                print(f"  [Error] Unexpected error parsing date '{date_str}': {e}")
                parse_errors += 1

        if year is not None and year in [2024, 2025]:
            filtered_answers_data.append(item)
            kept_count += 1
        else:
            skipped_count += 1

    print(f"\nFiltering Complete:")
    print(f"  - Kept {kept_count} records (Year 2024 or 2025).")
    print(f"  - Skipped {skipped_count} records (Other years or N/A/unparseable date).")
    if parse_errors > 0:
        print(f"  - Encountered {parse_errors} date parsing errors.")
else:
    print("[Error] 'all_answers_data' not found or is not a list. Cannot filter.")
    filtered_answers_data = []

# Now 'filtered_answers_data' holds only the records from 2024 and 2025
# The next cell (Cell 7) should use this list.


--- Filtering Data by Year (2024-2025) ---
Processing 586 records...

Filtering Complete:
  - Kept 213 records (Year 2024 or 2025).
  - Skipped 373 records (Other years or N/A/unparseable date).


In [7]:
# -*- coding: utf-8 -*-
# --- Cell 7: Save Collected Data to CSV File ---
# (Code remains the same as in your provided notebook)

print(f"\n--- Saving Filtered Data ---\n")

if 'filtered_answers_data' in locals() and filtered_answers_data:
    print(f"Attempting to save {len(filtered_answers_data)} filtered records to {OUTPUT_FILENAME}...")
    try:
        df_final = pd.DataFrame(filtered_answers_data)[['Question_Log_URL', 'Full_Answer_Text', 'Date']]
        df_final.to_csv(OUTPUT_FILENAME, index=False, encoding='utf-8-sig')
        print(f"[SUCCESS] Filtered data successfully saved to {OUTPUT_FILENAME}")
    except KeyError as ke:
         print(f"[ERROR] Failed to create DataFrame. Missing key: {ke}. Check data structure in 'filtered_answers_data'.")
    except Exception as e:
        print(f"[ERROR] Failed to save filtered data to CSV: {e}")
        print(f"        Check if '{OUTPUT_FILENAME}' is open in another program or if you have write permissions.")
elif 'all_answers_data' in locals() and all_answers_data and not filtered_answers_data:
     print(f"\nNo records found matching the year filter (2024-2025). CSV file '{OUTPUT_FILENAME}' not created/updated with filtered data.")
else:
    print("\nNo data collected or list is empty, CSV file not created.")

print("\n--- Script End ---")


--- Saving Filtered Data ---

Attempting to save 213 filtered records to edx_reviews_log.csv...
[SUCCESS] Filtered data successfully saved to edx_reviews_log.csv

--- Script End ---
